# Cloud Infrastructure Auto-Scaler — Model Notebook
**Author:** Manasa | KL University Hyderabad

Covers: telemetry simulation, EDA, LSTM architecture, cost-function analysis.


In [ ]:
import sys
sys.path.insert(0, '../data')
sys.path.insert(0, '../backend')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('Ready')

## 1. Telemetry Simulation — Single Node 24h Profile

In [ ]:
from telemetry_simulator import generate_node_telemetry

df = generate_node_telemetry('node-demo', n_points=1440, base_cpu=45)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(df['cpu_pct'], color='#00b4ff', linewidth=0.8)
axes[0].axhline(85, color='red', linestyle='--', alpha=0.5, label='Overload threshold')
axes[0].set_ylabel('CPU %')
axes[0].set_title('24h Server Telemetry — node-demo')
axes[0].legend()

axes[1].plot(df['memory_pct'], color='#00ffcc', linewidth=0.8)
axes[1].set_ylabel('Memory %')

axes[2].fill_between(range(len(df)), df['network_mbps'], alpha=0.4, color='mediumpurple')
axes[2].set_ylabel('Network MB/s')
axes[2].set_xlabel('Minute')

# Mark anomalies on CPU plot
anomaly_idx = df[df['is_anomaly']==1].index
axes[0].scatter(anomaly_idx, df.loc[anomaly_idx, 'cpu_pct'], color='red', s=20, zorder=5, label='Anomaly')

plt.tight_layout()
plt.show()
print(f'Anomalies: {df["is_anomaly"].sum()} / {len(df)} minutes ({df["is_anomaly"].mean()*100:.1f}%)')

## 2. Multi-Node Cluster Overview

In [ ]:
from telemetry_simulator import get_live_node_snapshot

snap = get_live_node_snapshot(100)
snap_df = pd.DataFrame(snap)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# CPU distribution
axes[0].hist(snap_df['cpu_pct'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(85, color='red', linestyle='--', label='Overload')
axes[0].axvline(70, color='orange', linestyle='--', label='Warning')
axes[0].set_title('CPU Distribution — 100 Nodes')
axes[0].set_xlabel('CPU %')
axes[0].legend()

# Status breakdown
status_counts = snap_df['status'].value_counts()
colors = {'healthy': '#00e676', 'warning': '#ffab40', 'overload': '#ff1744'}
axes[1].bar(status_counts.index, status_counts.values,
            color=[colors.get(s,'grey') for s in status_counts.index])
axes[1].set_title('Node Status Breakdown')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()
print(snap_df['status'].value_counts())

## 3. LSTM Training Sequence Visualisation

In [ ]:
from telemetry_simulator import generate_training_sequences

X, y = generate_training_sequences(n_sequences=200)
print(f'X: {X.shape}  y: {y.shape}')

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.plot(X[i, :, 0], label='CPU', color='#00b4ff')
    ax.plot(X[i, :, 1], label='Memory', color='#00ffcc', alpha=0.7)
    ax.axhline(y[i], color='red', linestyle='--', alpha=0.7, label=f'Target peak: {y[i]:.1f}%')
    ax.set_title(f'Sequence {i+1}')
    if i == 0:
        ax.legend(fontsize=7)

plt.suptitle('Sample LSTM Input Sequences (60-min windows)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Cost-Function Evaluator Analysis

In [ ]:
from train_lstm import cost_function_evaluator

cpu_levels = np.arange(10, 100, 5)
costs, latencies, recs = [], [], []

for cpu in cpu_levels:
    r = cost_function_evaluator(float(cpu), n_nodes=100)
    costs.append(r['total_cost_score'])
    latencies.append(r['latency_overhead_ms'])
    recs.append(r['recommendation'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cpu_levels, costs, marker='o', color='#7c4dff')
axes[0].set_title('Total Cost Score vs Predicted CPU')
axes[0].set_xlabel('Predicted Peak CPU (%)')
axes[0].set_ylabel('Cost Score (USD)')

axes[1].plot(cpu_levels, latencies, marker='s', color='#ff6b35')
axes[1].set_title('Estimated Latency Overhead vs CPU')
axes[1].set_xlabel('Predicted Peak CPU (%)')
axes[1].set_ylabel('Latency Overhead (ms)')

plt.tight_layout()
plt.show()

# Show recommendation transitions
for cpu, rec in zip(cpu_levels, recs):
    print(f'CPU={cpu:3.0f}% → {rec}')